# ASL Colab Gradio Demo

Temporary operator-run demo for single-video Top-50 ASL inference.

Run path:
1. Select a CUDA GPU runtime.
2. Run setup.
3. Load the combined Top-50 Gemma-4 LoRA with Unsloth.
4. Launch Gradio with `share=True`.
5. Upload one ASL video and inspect the prediction/debug JSON.

This notebook is not a durable hosted app. It is meant to prove the model path works in Colab.

## 1. Runtime setup

Use a GPU runtime. A100/H100-class hardware is preferred for the 26B/A4B adapter.

In [ ]:
# Runtime setup
import os

os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "60")
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "600")

!pip -q install opencv-python-headless gradio huggingface_hub
!pip -q install --upgrade --no-cache-dir unsloth

## 2. Config

Defaults are training-parity: 30 evenly sampled RGB frames resized to 448x448.

The allowlist below is the WLASL/model Top-50 label space used by the current notebooks, not the older conversational demo list.

In [ ]:
# Demo config
ADAPTER_REPO = "AlexD281/asl-gemma4-26b-a4b-zahid-wlasl-combined-top50-lora"
NUM_FRAMES = 30
FRAME_SIZE = 448
MAX_SEQ_LENGTH = 8192
MAX_NEW_TOKENS = 12
MAX_CLIP_SECONDS = 8.0

TOP50_LABELS = [
    "drink",
    "like",
    "wrong",
    "forget",
    "computer",
    "finish",
    "hot",
    "mother",
    "now",
    "orange",
    "hearing",
    "color",
    "birthday",
    "need",
    "book",
    "before",
    "deaf",
    "fine",
    "no",
    "yes",
    "all",
    "black",
    "kiss",
    "study",
    "white",
    "dance",
    "but",
    "cook",
    "paper",
    "visit",
    "door",
    "college",
    "your",
    "use",
    "better",
    "last year",
    "will",
    "chair",
    "clothes",
    "candy",
    "year",
    "many",
    "woman",
    "blue",
    "fish",
    "hat",
    "bird",
    "cow",
    "enjoy",
    "meet",
]
ALIASES = {
    "thank you": "thanks",
    "thank-you": "thanks",
    "dont like": "don't like",
    "do not like": "don't like",
    "dont understand": "don't understand",
    "do not understand": "don't understand",
    "dont know": "don't know",
    "do not know": "don't know",
}

assert len(TOP50_LABELS) == 50
assert len(set(TOP50_LABELS)) == 50
print("Adapter:", ADAPTER_REPO)
print("Frame contract:", NUM_FRAMES, "frames @", FRAME_SIZE, "px")
print("Top-50 labels:", len(TOP50_LABELS))

## 3. Shared helpers

The output guardrail is strict: only canonical Top-50 labels or approved aliases become accepted predictions. If the model says another real-looking gloss, the notebook shows it as the rejected candidate instead of hiding it behind a blank `null` value.

In [ ]:
# Shared helpers
from __future__ import annotations

import gc
import json
import math
import re
import traceback
from pathlib import Path
from typing import Any

import cv2
import torch
from PIL import Image

TOP50_SET = set(TOP50_LABELS)


def normalize_gloss(text: str) -> str:
    normalized = str(text).strip().lower()
    normalized = re.sub(r"^`+|`+$", "", normalized)
    normalized = re.sub(r"[^a-z0-9_ '\-]+", "", normalized)
    normalized = re.sub(r"\s+", " ", normalized).strip()
    return normalized


def strict_top50_prediction(raw_output: str) -> dict[str, Any]:
    lines = [line.strip() for line in str(raw_output).splitlines() if line.strip()]
    first_line = lines[0] if lines else ""
    normalized_candidate = normalize_gloss(first_line)
    candidate = ALIASES.get(normalized_candidate, normalized_candidate)
    if candidate in TOP50_SET:
        return {
            "status": "ok",
            "prediction": candidate,
            "candidate_gloss": candidate,
            "validation_reason": "candidate is in the approved Top-50 label set",
            "raw_output": raw_output,
        }
    return {
        "status": "out_of_allowlist",
        "prediction": None,
        "candidate_gloss": candidate or None,
        "validation_reason": "model output is not in the approved Top-50 label set",
        "raw_output": raw_output,
    }


def build_prompt() -> str:
    return (
        "Identify the ASL sign shown across these frames. "
        "Return exactly one gloss label from the approved list.\n"
        f"Approved labels: {', '.join(TOP50_LABELS)}"
    )


def compact_error(exc: BaseException) -> dict[str, Any]:
    return {
        "type": exc.__class__.__name__,
        "message": str(exc),
        "traceback_tail": "".join(traceback.format_exception(type(exc), exc, exc.__traceback__)[-4:]),
    }


def gpu_diagnostics() -> dict[str, Any]:
    info = {
        "torch_version": getattr(torch, "__version__", "unknown"),
        "cuda_available": bool(torch.cuda.is_available()),
        "adapter_repo": ADAPTER_REPO,
        "num_frames": NUM_FRAMES,
        "frame_size": FRAME_SIZE,
        "max_seq_length": MAX_SEQ_LENGTH,
    }
    if torch.cuda.is_available():
        index = torch.cuda.current_device()
        props = torch.cuda.get_device_properties(index)
        info.update({
            "cuda_device_name": torch.cuda.get_device_name(index),
            "cuda_total_vram_gb": round(props.total_memory / (1024 ** 3), 3),
            "cuda_compute_capability": f"{props.major}.{props.minor}",
        })
    return info


def frame_indices(total_frames: int, count: int) -> list[int]:
    if count <= 1:
        return [0]
    if total_frames <= 0:
        return list(range(count))
    last = max(total_frames - 1, 0)
    return [round(i * last / (count - 1)) for i in range(count)]


def sample_video_frames(video_path: str | Path) -> list[Image.Image]:
    path = Path(video_path)
    if not path.exists():
        raise FileNotFoundError(f"Video not found: {path}")

    capture = cv2.VideoCapture(str(path))
    if not capture.isOpened():
        raise ValueError(f"Could not open video: {path}")

    total_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    fps = float(capture.get(cv2.CAP_PROP_FPS) or 0.0)
    if total_frames > 0 and fps > 0:
        duration = total_frames / fps
        if duration > MAX_CLIP_SECONDS:
            capture.release()
            raise ValueError(f"Video is {duration:.2f}s; maximum allowed is {MAX_CLIP_SECONDS:.2f}s.")

    frames: list[Image.Image] = []
    last_good = None
    for index in frame_indices(total_frames, NUM_FRAMES):
        if total_frames > 0:
            capture.set(cv2.CAP_PROP_POS_FRAMES, index)
        ok, frame = capture.read()
        if not ok or frame is None:
            if last_good is None:
                continue
            frame = last_good
        else:
            last_good = frame
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image = Image.fromarray(rgb).convert("RGB").resize((FRAME_SIZE, FRAME_SIZE), Image.Resampling.BICUBIC)
        frames.append(image)

    capture.release()
    if not frames:
        raise ValueError("No readable frames were extracted from the video.")
    while len(frames) < NUM_FRAMES:
        frames.append(frames[-1].copy())
    return frames[:NUM_FRAMES]

print(json.dumps(gpu_diagnostics(), indent=2))

## 4. Load model

This mirrors the known-good notebook path: download adapter snapshot, load local snapshot with `FastVisionModel.from_pretrained`, then call `FastVisionModel.for_inference`.

In [ ]:
# Load model with Unsloth FastVisionModel
from huggingface_hub import login, snapshot_download
from unsloth import FastVisionModel

hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token, add_to_git_credential=False)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU runtime is required for this 26B/A4B demo.")

print(json.dumps(gpu_diagnostics(), indent=2))

gc.collect()
torch.cuda.empty_cache()

adapter_local = snapshot_download(ADAPTER_REPO, token=hf_token)
print("Adapter local path:", adapter_local)

model, tokenizer = FastVisionModel.from_pretrained(
    model_name=adapter_local,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
    max_seq_length=MAX_SEQ_LENGTH,
    device_map={"": 0},
)
FastVisionModel.for_inference(model)
print("Model ready")

## 5. Prediction function

This uses notebook-style chat-template inference: text prompt first, then image frames.

In [ ]:
# Prediction function
def predict_video(video_path: str | None):
    if not video_path:
        return "Upload one ASL video first.", {"status": "missing_video"}

    try:
        frames = sample_video_frames(video_path)
        prompt = build_prompt()
        user_msg = {
            "role": "user",
            "content": [{"type": "text", "text": prompt}] + [
                {"type": "image", "image": image} for image in frames
            ],
        }
        inputs = tokenizer.apply_chat_template(
            [user_msg],
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        )
        if hasattr(inputs, "to"):
            inputs = inputs.to(model.device)
        else:
            inputs = {key: value.to(model.device) if hasattr(value, "to") else value for key, value in inputs.items()}

        prompt_len = int(inputs["input_ids"].shape[-1])
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        raw_output = tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True).strip()
        prediction = strict_top50_prediction(raw_output)
        debug = {
            **prediction,
            "adapter_repo": ADAPTER_REPO,
            "frame_count": len(frames),
            "frame_size": FRAME_SIZE,
            "max_seq_length": MAX_SEQ_LENGTH,
            "gpu": gpu_diagnostics(),
        }
        if prediction["status"] == "ok":
            summary = (
                f"Accepted prediction: {prediction['prediction']}\n"
                f"Model candidate: {prediction['candidate_gloss']}\n"
                f"Status: ok\n"
                f"Reason: {prediction['validation_reason']}\n"
                f"Raw output: {raw_output}"
            )
        else:
            summary = (
                "Accepted prediction: none\n"
                f"Model candidate: {prediction['candidate_gloss'] or '(empty)'}\n"
                f"Status: {prediction['status']}\n"
                f"Reason: {prediction['validation_reason']}\n"
                f"Raw output: {raw_output}"
            )
        return summary, debug
    except Exception as exc:
        debug = {
            "status": "error",
            "error": compact_error(exc),
            "gpu": gpu_diagnostics(),
            "adapter_repo": ADAPTER_REPO,
            "frame_count": NUM_FRAMES,
            "frame_size": FRAME_SIZE,
            "max_seq_length": MAX_SEQ_LENGTH,
        }
        summary = f"Inference failed: {type(exc).__name__}: {exc}"
        return summary, debug

## 6. Launch Gradio

The public `share=True` URL is temporary and lives only while this Colab runtime is active.

In [ ]:
# Launch Gradio demo
import gradio as gr

with gr.Blocks(title="ASL Top-50 Colab Demo") as demo:
    gr.Markdown("# ASL Top-50 Colab Demo")
    gr.Markdown(
        f"Adapter: `{ADAPTER_REPO}`  \n"
        f"Frame contract: {NUM_FRAMES} frames at {FRAME_SIZE}x{FRAME_SIZE}  \n"
        "Output guardrail: strict Top-50; out-of-list model candidates are shown but not accepted."
    )
    video = gr.Video(label="Upload one ASL video", sources=["upload"], format=None)
    submit = gr.Button("Predict", variant="primary")
    result = gr.Textbox(label="Prediction", lines=4)
    debug = gr.JSON(label="Debug")
    submit.click(fn=predict_video, inputs=video, outputs=[result, debug])

demo.launch(share=True, debug=True)